In [1]:
from pathlib import Path
root = Path.cwd()

#import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import netCDF4
import numcodecs
import rioxarray
# import rasterio
import shutil
import zarr
import dask
from numba import njit, prange

import dask.array as da
from dask.distributed import Client, LocalCluster

#from skimage.feature.texture import graycomatrix, graycoprops

In [2]:
def get_quantiles(s, month_list):
    ss = s.sel(band=['B08','B04'],time=s.time.dt.month.isin(month_list))

    tdvi_alldates = (1.5*((ss.sel(band='B08')-ss.sel(band='B04'))/(ss.sel(band='B08')**2+ss.sel(band='B04')+0.5)**0.5))
        
    tdvi = tdvi_alldates.median(dim='time',skipna=True)

    q = tdvi.quantile([0.2,0.98],skipna=True).compute()
    out = {'q2': q.sel(quantile=0.2).drop_vars('quantile'), 'q98':q.sel(quantile=0.98).drop_vars('quantile')}

    print(f'q2: {out['q2']}, q98: {out['q98']}')    

# april/may: -0.01 - 0.45
# june/july: -0.01 - 0.73
# oct/nov: -0.02 - 0.51

def make_grayscale(raster_ts,months,levels):

    ss = raster_ts.sel(band='tdv',time=raster_ts.time.dt.month.isin(months))
    
    tdvi = ss.median(dim='time',skipna=True)
    
    TDVI_LO, TDVI_HI = -0.02, 0.8  # set once for all images based on nyc18 2nd & 98th percentiles
    scaled = ((tdvi - TDVI_LO) / (TDVI_HI - TDVI_LO)).clip(0, 1) * (levels - 1)

    # this method of scaling is less robust and also different for each year and season, which reduces transferability.
    # mn = float(tdvi.min(skipna=True).compute())
    # mx = float(tdvi.max(skipna=True).compute())
    # scaled = (tdvi - mn)/(mx-mn) * (levels-1)
    #valid_mask = np.isfinite(tdvi)

    valid_mask = xr.DataArray(
        da.isfinite(tdvi.data),
        dims=tdvi.dims,
        coords=tdvi.coords
    )

    #valid_mask = valid_mask.chunk({'time': 1, 'y': spatial_chunk, 'x': spatial_chunk})
    quantized = xr.where(valid_mask,xr.apply_ufunc(np.rint,scaled,dask='parallelized').clip(0,levels-1).astype('uint8'),0)

    return quantized, valid_mask

@njit(cache=True)
def build_glcm(window, levels, distances, angles):
    """
    Returns normalised symmetric GLCM: shape (levels, levels, n_dist, n_angles)
    """
    n_dist = len(distances)
    n_ang = len(angles)
    g = np.zeros((levels, levels, n_dist, n_ang), dtype=np.float32)
    h, w = window.shape

    for d_idx in range(n_dist):
        d = distances[d_idx]
        for a_idx in range(n_ang):
            angle = angles[a_idx]
            dy = int(round(-d * np.sin(angle)))
            dx = int(round( d * np.cos(angle)))
            for y in range(h):
                for x in range(w):
                    ny = y + dy
                    nx = x + dx
                    if 0 <= ny < h and 0 <= nx < w:
                        i_val = window[y, x]
                        j_val = window[ny, nx]
                        g[i_val, j_val, d_idx, a_idx] += 1.0
                        g[j_val, i_val, d_idx, a_idx] += 1.0  # symmetric

    # normalise each (dist, angle) slice
    for d_idx in range(n_dist):
        for a_idx in range(n_ang):
            total = g[:, :, d_idx, a_idx].sum()
            if total > 0:
                for i in range(levels):
                    for j in range(levels):
                        g[i, j, d_idx, a_idx] /= total
    return g


@njit(cache=True)
def glcm_metrics(g, levels):
    """
    Compute contrast, dissimilarity, homogeneity, ASM, correlation,
    mean, entropy — averaged over distances and angles.
    Returns 1D array of length 8 in that order.
    """
    n_dist = g.shape[2]
    n_ang  = g.shape[3]
    n_slices = n_dist * n_ang

    contrast     = 0.0
    dissimilarity= 0.0
    homogeneity  = 0.0
    asm          = 0.0
    variance = 0.0
    #mean_val     = 0.0
    entropy_val  = 0.0
    # correlation needs marginal stats
    #corr         = 0.0

    for d in range(n_dist):
        for a in range(n_ang):
            # marginal means and stds for variance
            mu_i = 0.0
            mu_j = 0.0
            for i in range(levels):
                for j in range(levels):
                    mu_i += i * g[i, j, d, a]
                    mu_j += j * g[i, j, d, a]
            sig_i = 0.0
            sig_j = 0.0
            for i in range(levels):
                for j in range(levels):
                    sig_i += (i - mu_i) ** 2 * g[i, j, d, a]
                    sig_j += (j - mu_j) ** 2 * g[i, j, d, a]
            sig_i = np.sqrt(sig_i)
            sig_j = np.sqrt(sig_j)
            variance = sig_i * sig_i  ## average across angles, not levels

            for i in range(levels):
                for j in range(levels):
                    p = g[i, j, d, a]
                    if p == 0.0:
                        continue
                    ## formulas from Hall-Beyer (2017)
                    #variance += sig_i ** 2
                    diff = float(i - j)
                    contrast      += diff * diff * p
                    dissimilarity += abs(diff) * p
                    homogeneity   += p / (1.0 + diff * diff)
                    asm           += p * p
                    #mean_val      += i * p
                    entropy_val   -= p * np.log2(p)
                    #if sig_i > 0 and sig_j > 0:
                        #corr += ((i - mu_i) * (j - mu_j) * p) / (sig_i * sig_j)

    # average over all distance/angle combinations
    contrast      /= n_slices
    dissimilarity /= n_slices
    homogeneity   /= n_slices
    asm           /= n_slices
    variance      /= n_slices  
    #mean_val      /= n_slices
    entropy_val   /= n_slices
    #corr          /= n_slices

    out = np.empty(6, dtype=np.float32)
    out[0] = contrast
    out[1] = dissimilarity
    out[2] = homogeneity
    out[3] = asm
    out[4] = variance
    #out[5] = corr
    #out[6] = mean_val
    out[5] = entropy_val
    return out


@njit(cache=True, parallel=True)
def calculate_metrics_numba(block, valid_mask, levels, pad,
                             distances, angles):
    """
    block, valid_mask: 2D np arrays (h, w)
    returns: (7, h-2*pad, w-2*pad) float32
    """
    h, w = block.shape
    oh = h - 2 * pad
    ow = w - 2 * pad
    out = np.full((6, oh, ow), np.nan, dtype=np.float32)

    for i in prange(pad, h - pad):           # parallel over rows
        for j in range(pad, w - pad):
            win_mask = valid_mask[i-pad:i+pad+1, j-pad:j+pad+1]
            all_valid = True
            for mi in range(win_mask.shape[0]):
                for mj in range(win_mask.shape[1]):
                    if not win_mask[mi, mj]:
                        all_valid = False
                        break
                if not all_valid:
                    break
            if not all_valid:
                continue

            window = block[i-pad:i+pad+1, j-pad:j+pad+1]

            # check uniform window
            mn = window[0, 0]
            uniform = True
            for wi in range(window.shape[0]):
                for wj in range(window.shape[1]):
                    if window[wi, wj] != mn:
                        uniform = False
                        break
                if not uniform:
                    break
            if uniform:
                # fill with known analytical values
                # contrast, dissimilarity
                out[0, i - pad, j - pad] = 0.0
                out[1, i - pad, j - pad] = 0.0
                # homogeneity, ASM, variance
                out[2, i - pad, j - pad] = 1.0
                out[3, i - pad, j - pad] = 1.0
                out[4, i - pad, j - pad] = 0.0
                # correlation — undefined
                #out[5, i - pad, j - pad] = 1.0
                # mean = the single grey level value
                #out[6, i - pad, j - pad] = np.float32(mn)
                # entropy
                out[5, i - pad, j - pad] = 0.0

                continue

            g = build_glcm(window, levels, distances, angles)
            metrics = glcm_metrics(g, levels)

            for k in range(6):
                out[k, i - pad, j - pad] = metrics[k]

    return out

def calculate_metrics_for_block(block, valid_mask, levels, pad, distances, angles):
   
    distances_arr = np.array(distances, dtype=np.int64)
    angles_arr    = np.array(angles,    dtype=np.float64)
    return calculate_metrics_numba(
        block.astype(np.int32),
        valid_mask.astype(np.bool_),
        levels, pad,
        distances_arr, angles_arr
    )


def get_blockwise_glcm_metrics(tdvi_array,metrics_list,levels,window_size,angles,distances,valid_mask,out_path,spatial_chunk):

    pad = window_size // 2

    #n_times = tdvi_array.sizes['time'] ## array is now 2d (no time dim)
    n_metrics = len(metrics_list)
    img = tdvi_array.data # (x,y)
    msk = valid_mask.data
    
    ### initialize zarr store
    # template = xr.DataArray(
    #     da.empty((n_times, n_metrics, tdvi_array.sizes['y'], tdvi_array.sizes['x']),
    #              dtype=np.float32,
    #              chunks=(1, n_metrics, spatial_chunk, spatial_chunk)),
    #     dims=('time', 'metric', 'y', 'x'),
    #     coords={
    #         'time':   tdvi_array.time,
    #         'metric': metrics_list,
    #         'y':      tdvi_array.y,
    #         'x':      tdvi_array.x
    #     }
    # ).to_dataset(name='glcm')
   
   

    #template.to_zarr(out_path, mode='w', encoding=encoding, zarr_format=2,compute=False)

    #time_steps = []
    
    #for i, t in enumerate(tdvi_array.time.values):

    result = da.map_overlap(calculate_metrics_for_block,
                                img,msk,
                                distances=tuple(distances),
                                angles=tuple(angles),
                                levels=levels,
                                pad=int(pad),
                                new_axis=0, ## add new axis for metrics
                                chunks=(len(metrics_list),) + img.chunks, 
                                depth={0:pad,1:pad},
                                trim=False,
                                dtype=np.float32,
                                boundary='reflect') 
    
    out_da = xr.DataArray(result,dims=('band','y','x'),coords={'band':metrics_list,'y':tdvi_array.y,'x':tdvi_array.x}).to_dataset(name='glcm')

    encoding = {
        'glcm': {
            'chunks': (n_metrics, spatial_chunk, spatial_chunk),
            'compressor': numcodecs.Blosc(cname='zstd', clevel=3, shuffle=numcodecs.Blosc.SHUFFLE)
        }
    } 

    out_da.to_zarr(out_path,mode='w',encoding=encoding,zarr_format=2)

  

In [3]:
cluster = LocalCluster(n_workers=4,threads_per_worker=1,processes=True,memory_limit='6GB')

client = Client(cluster)

client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 4,Total memory: 22.35 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:56116,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:56141,Total threads: 1
Dashboard: http://127.0.0.1:56142/status,Memory: 5.59 GiB
Nanny: tcp://127.0.0.1:56119,


In [4]:
## check chunk sizes
siteyear = 'nyc21'
all_data = xr.open_zarr(root / 'data' / siteyear / f'{siteyear}_allbandsandindices.zarr')['variables']

bytes = all_data.chunks[0][0] * all_data.chunks[1][0] *  all_data.chunks[2][0] *  all_data.chunks[3][0] *  4

mb = bytes/1e6

print(f'MB per chunk: {mb}')

MB per chunk: 12.845056


In [ ]:
siteyear = 'nyc21'
metrics_list = ['contrast', 'dissimilarity', 'homogeneity',
                'ASM', 'variance','entropy']


angles = tuple([0, np.pi/4, np.pi/2, 3*np.pi/4])
distances = tuple([1])
levels = 8
windowsize = 3

#chunks = {'time':-1,'band':9,'y':256,'x':256}
# s = xr.open_dataarray(root / 'data' / siteyear / f'{siteyear}_bands2348_ARD_buffered.nc',chunks={'time':1,'y':spatial_chunk,'x':spatial_chunk})
##  use non-clipped data to avoid edge shrinkage
s = xr.open_zarr(root / 'data' / siteyear / f'{siteyear}_allbandsandindices.zarr')['variables']
# s = xr.open_zarr(root / 'data' / siteyear / f'{siteyear}_daily_raw_10m_aligned.zarr')['reflectance']
#s = s.chunk(chunks)
spatial_chunk = s.chunksizes['y'][0]
########### set months ##############################
# spring = [3,4,5], summer = [6,7,8], autumn = [9,10,11]
season_months = {'spring':[4,5],'summer':[6,7],'fall':[10,11]}

a_dict = {'spring': None, 'summer': None, 'fall':None}
for season_label, month_list in season_months.items():
    # season_label = 'spring'
    # month_list = [3,4,5]

    out_path = root / 'data' / siteyear / f'{siteyear}_{season_label}_glcm.zarr'
    if out_path.exists():
        shutil.rmtree(out_path)

    q, v = make_grayscale(s,month_list,levels=levels)
    ##################################################

    get_blockwise_glcm_metrics(tdvi_array=q,valid_mask=v,metrics_list=metrics_list,angles=angles,distances=distances,levels=levels,window_size=windowsize,out_path=out_path,spatial_chunk=spatial_chunk)


    a_dict[season_label] = xr.open_zarr(out_path)['glcm']

## get seasonal deltas and append to seasonal stats zarr store

def add_band_suffix(array, suff):
    return array.assign_coords({'band': (array['band'].astype('str') + f'_{suff}').astype(object)})
    
spring_delta = add_band_suffix(a_dict['summer'] - a_dict['spring'], suff='spring_delta')
fall_delta = add_band_suffix(a_dict['summer'] - a_dict['fall'],suff='fall_delta')

out_list = [add_band_suffix(a_dict['spring'],'spring'), add_band_suffix(a_dict['summer'],'summer'),add_band_suffix(a_dict['fall'],'fall'), spring_delta, fall_delta]

all_data = xr.concat(out_list,dim='band')

all_data = all_data.to_dataset(name='features')

store_path = root / 'data' / siteyear / f'{siteyear}_seasonalstats.zarr'
w = xr.open_zarr(store_path)['features']
band_chunk = w.chunksizes['band'][0]
spatial_chunk = w.chunksizes['y'][0]


boundary = gpd.read_file(root / 'data' / 'nyc_boundary.gpkg')
all_data = all_data.rio.write_crs(boundary.crs).rio.set_spatial_dims(x_dim='x',y_dim='y').rio.write_coordinate_system()
all_data = all_data.rio.clip(boundary.geometry)

all_data = all_data.chunk({'band':band_chunk,'y':spatial_chunk,'x':spatial_chunk})
all_data.to_zarr(store_path,mode='a',append_dim='band',zarr_format=2)


#### archive

In [ ]:
cluster = LocalCluster(n_workers=4,threads_per_worker=1,processes=True,memory_limit='6GB')

client = Client(cluster)

client

In [ ]:
## truncate zarr store using resize oops
out_path = root / 'data' / 'nyc21' / 'nyc21_seasonalstats_dsen2.zarr'
store = zarr.open(out_path,mode='r+')

store['features'].resize((920,store['features'].shape[1],store['features'].shape[2]))
store['band'].resize((920,))
zarr.consolidate_metadata(out_path) ## update metadata with new size

<Group file://c:/Users/roseh/Documents/utc_paper/data/nyc21/nyc21_seasonalstats_dsen2.zarr>

In [57]:
siteyear = 'nyc18'
band_method = 'resampled'
### get chunk sizes
out_path = root / 'data' / siteyear / f'{siteyear}_seasonalstats.zarr'
store = zarr.open(out_path,mode='r')
store_chunks = store['features'].chunks

for season_name in ['spring','summer','autumn']:

    ds = xr.open_zarr(root / 'data' / siteyear / f'{siteyear}_{season_name}_glcm.zarr')
    season = ds[list(ds.data_vars)[0]]


    # aggregations
    mean = season.mean(dim='time',skipna=True).reset_coords(drop=True)
    mx = season.max(dim='time',skipna=True).reset_coords(drop=True)
    mn = season.min(dim='time',skipna=True).reset_coords(drop=True)
    std = season.std(dim='time',skipna=True).reset_coords(drop=True)

    amp = mx - mn
    cv = std / mean.clip(min=1e-6)

    q = season.quantile([0.25, 0.5, 0.75], dim="time", skipna=True)
    p25 = q.sel(quantile=0.25).drop_vars("quantile")
    p50 = q.sel(quantile=0.5 ).drop_vars("quantile")
    p75 = q.sel(quantile=0.75).drop_vars("quantile")
    iqr = p75 - p25

    stats = {'mean':mean,
                'max':mx,
                'min':mn,
                'std':std,
                'amp':amp,
                'cv':cv,
                'p25':p25,
                'p50':p50,
                'p75':p75,
                'iqr':iqr}

    ## add new band coordinate to each stat array
    segments = []
    for st in stats:
        a = stats[st]
        new_band = a['metric'].str[:3].astype('str') + f'_{season_name}_{st}'
        a = a.assign_coords({'metric':new_band}).rename({'metric':'band'})
        segments.append(a)

    all_segs = xr.concat(segments,dim='band')
    all_segs = all_segs.to_dataset(name='features').astype('float32')

    band_chunk = store_chunks[0]
    spatial_chunk = store_chunks[1]

    ## dask chunks have to match zarr encoding
    all_segs = all_segs.chunk({'band':band_chunk,'y':spatial_chunk,'x':spatial_chunk})

    
    all_segs.to_zarr(out_path,mode='a',append_dim='band')


    

#### Test numba version on subset

In [ ]:
def get_most_valid(a):
    vf = a.notnull().mean(("y","x"))
    idx = vf.idxmax("time").values[0]
    return a.sel(time=idx).squeeze(drop=True)

def get_tdvi_patch(raster, pp):
    patch = raster.rio.clip(pp)
    tdvi = (1.5*((patch.sel(band='B08')-patch.sel(band='B04'))/(patch.sel(band='B08')**2+patch.sel(band='B04')+0.5)**0.5))
    return tdvi.compute()


# scale image and convert to int
def make_grayscale(tdvi_array,levels):

    tdvi = tdvi_array.values
    valid = np.isfinite(tdvi)
    mn = np.nanmin(tdvi)
    mx = np.nanmax(tdvi)
    scaled = ((tdvi - mn)/(mx-mn) * (levels-1))
    
    # 1D
    quantized = np.clip(np.rint(scaled[valid]),0,levels-1).astype('uint8')
    output = np.full(tdvi.shape,0,dtype=np.uint8)
    output[valid] = quantized

    return output, valid                                


s = xr.open_dataarray(root / 'data' / 'nyc21' / 'nyc21_bands2348_ARD.nc',chunks={'time':1,'y':512,'x':512})
p = gpd.read_file(root / 'data' / 'glcm_patch.gpkg')

# get one july date
ss = s.sel(band=['B08','B04'],time=s.time.dt.month.isin([7]))
img = get_most_valid(ss)
tdvi_bk = get_tdvi_patch(img,p.iloc[1])
tdvi_si = get_tdvi_patch(img,p.iloc[0])

# may - jun
spring = s.sel(band=['B08','B04'],time=s.time.dt.month.isin([5,6]))
tdvi_spring_bk = get_tdvi_patch(spring,p.iloc[1])
tdvi_spring_si = get_tdvi_patch(spring,p.iloc[0])

# jul - aug
summer = s.sel(band=['B08','B04'],time=s.time.dt.month.isin([7,8]))
tdvi_summer_bk = get_tdvi_patch(summer,p.iloc[1])
tdvi_summer_si = get_tdvi_patch(summer,p.iloc[0])

bk_gs, bk_valid = make_grayscale(tdvi_bk,levels=32)
si_gs, si_valid = make_grayscale(tdvi_si,levels=32)

block, valid_mask = make_grayscale(tdvi_si,levels=32)

# set glcm parameters
angles = tuple([0, np.pi/4, np.pi/2, 3*np.pi/4])
distances = tuple([1])
levels = 32
windowsize = 5

pad = windowsize // 2

distances_arr = np.array(distances, dtype=np.int64)
angles_arr    = np.array(angles,    dtype=np.float64)

out_si = calculate_metrics_numba(block=block.astype(np.int32),valid_mask=valid_mask.astype(np.bool_),levels=32,pad=pad,distances=distances_arr,angles=angles_arr)

#### Test skimage version on subset

In [ ]:
#### skimage version
# set glcm parameters
angles = tuple([0, np.pi/4, np.pi/2, 3*np.pi/4])
distances = tuple([1])
levels = 32
windowsize = 5

pad = windowsize // 2


metrics_list = ['contrast', 'dissimilarity', 'homogeneity',
                'ASM', 'energy', 'correlation', 'mean', 'entropy']

block, valid_mask = make_grayscale(tdvi_si,levels=32) ####### input raster


h,w = block.shape
n = len(metrics_list)
out = np.full((n, h, w), np.nan, dtype=np.float32)

for i in range(pad,h-pad):
    for j in range(pad,w-pad):
        #skip if nan pixels are included
        win_mask = valid_mask[i-pad:i+pad+1,j-pad:j+pad+1]
        if not win_mask.all():
            continue

        window = block[i-pad:i+pad+1,j-pad:j+pad+1]
        if window.min() == window.max():
            continue

        g = graycomatrix(window, distances=distances, angles=angles, levels=levels, 
                    normed=True, symmetric=True)
        
        for k, m in enumerate(metrics_list):
            result = float(graycoprops(g, m).mean())
            out[k,i,j] = result

#### Dask/skimage version - slow!

In [4]:

def calculate_metrics_for_block(block,valid_mask,metrics_list,levels,pad,angles,distances):
# pad to accomodate window size
    
    h,w = block.shape
    n = len(metrics_list)
    out = np.full((n, h, w), np.nan, dtype=np.float32)

    for i in range(pad, h - pad):
        for j in range(pad, w - pad):
            # skip if nan pixels are included
            win_mask = valid_mask[i-pad:i+pad+1,j-pad:j+pad+1]
            if not win_mask.all():
                continue

            window = block[i-pad:i+pad+1,j-pad:j+pad+1]
            if np.unique(window).size < 2:
                continue

            g = graycomatrix(window, distances=distances, angles=angles, levels=levels, 
                        normed=True, symmetric=True)
            
            for k, m in enumerate(metrics_list):
                result = float(graycoprops(g, m).mean())
                out[k,i,j] = result

    return out[:, pad:-pad, pad:-pad]


def get_blockwise_glcm_metrics(tdvi_array,metrics_list,levels,window_size,angles,distances,valid_mask,chunks={"time": 1, "y": 512, "x": 512}):

    pad = window_size // 2

    # tdvi_chunked = tdvi_array
    # mask_chunked = valid_mask.chunk(chunks)

    time_steps = []
    for t in tdvi_array.time.values:

        img = tdvi_array.sel(time=t).data # (x,y)
        msk = valid_mask.sel(time=t).data

        this_time = da.map_overlap(calculate_metrics_for_block,
                                    img,msk,
                                    metrics_list=tuple(metrics_list),
                                    distances=tuple(distances),
                                    angles=tuple(angles),
                                    levels=levels,
                                    pad=int(pad),
                                    new_axis=0, ## add new axis for metrics
                                    chunks=(len(metrics_list),) + img.chunks, 
                                    depth={0:pad,1:pad},
                                    
                                    trim=False,
                                    dtype=np.float32,
                                    boundary='reflect') 
        
        time_steps.append(this_time)

    out = da.stack(time_steps,axis=0)


    out_da = xr.DataArray(out,dims=('time','metric','y','x'),coords={'time':tdvi_array.time,'metric':metrics_list,'y':tdvi_array.y,'x':tdvi_array.x})


    return out_da.to_dataset(name='glcm')
   


In [ ]:
#medoids = xr.open_dataarray(root / 'data' / 'nyc17v6' / 'mostvalid_2017.nc')
spatial_chunk = 512
spring = s.sel(band=['B04','B08'], time=s.time.dt.month.isin([3,4,5]))

tdvi = (1.5*((spring.sel(band='B08')-spring.sel(band='B04'))/(spring.sel(band='B08')**2+spring.sel(band='B04')+0.5)**0.5)).chunk({'time':1,'y':spatial_chunk,'x':spatial_chunk}).persist()

# scale image and convert to int
levels = 42
mn = tdvi.min(skipna=True).compute()
mx = tdvi.max(skipna=True).compute()
scaled = (tdvi - mn)/(mx-mn) * (levels-1)
valid_mask = np.isfinite(tdvi)

valid_mask = valid_mask.chunk({'time': 1, 'y': spatial_chunk, 'x': spatial_chunk})
quantized = xr.where(valid_mask,xr.apply_ufunc(np.rint,scaled,dask='parallelized').clip(0,levels-1).astype('uint8'),0).chunk({'time': 1, 'y': spatial_chunk, 'x': spatial_chunk})

levels_glcm = levels
# set glcm parameters
angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
distances = [1]

metrics_list = ['homogeneity']
#metrics_list = ['contrast']

glcm = get_blockwise_glcm_metrics(tdvi_array=quantized,window_size=3,metrics_list=metrics_list,levels=levels_glcm,angles=angles,distances=distances,valid_mask=valid_mask)

encoding = {'glcm':{'chunks':(1,len(metrics_list),spatial_chunk,spatial_chunk),'compressor': zarr.Blosc(cname='zstd',clevel=3,shuffle=2)}}
glcm.to_zarr(root / 'data' / 'nyc21' / 'nyc21_spring_glcm.zarr',mode='w',encoding=encoding)